In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
import importlib, spk_feat_cluster_comp_analysis
importlib.reload(spk_feat_cluster_comp_analysis)
from spk_feat_cluster_comp_analysis import extract_eap_waveforms
from config import SPE1_DATA_ROOT, SPE1_PICKLE_ROOT, DICT_CHAN_PRED, NPX_FS

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Run-control flags ───────────────────────────────────────────────────────
FORCE_RERUN = False   # set True to recompute and overwrite pickles


# EAP waveform PCA — pre vs post temporal transition

For cells where spike cluster membership shows a strong temporal component (|Spearman ρ| > 0.3),
load the Neuropixels EAP recording and extract spike waveforms before and after the detected
changepoint. Run PCA on the waveforms and compare the pre/post distributions in PC space.

**Question**: does the extracellular spike waveform on the Neuropixels probe change at the same
transition point detected in the patch-clamp feature clustering?

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CLUSTER_PICKLE_DIR = SPE1_PICKLE_ROOT + '/cluster_pickles/'
NPX_DIR            = SPE1_DATA_ROOT  + '/filt_npx_recordings/'
TRANSITION_PICKLE  = CLUSTER_PICKLE_DIR + 'temporal_transitions.pkl'

# EAP window: -1 to +3 ms around spike peak at 30 kHz
PRE_MS   = 1.0   # ms before spike
POST_MS  = 3.0   # ms after spike
PRE_SAMP  = int(PRE_MS  * NPX_FS / 1000)
POST_SAMP = int(POST_MS * NPX_FS / 1000)
WIN_LEN   = PRE_SAMP + POST_SAMP   # samples per waveform

# Cells with NPX recordings available
NPX_CELLS = {f.replace('npx_filt.npy', '') for f in os.listdir(NPX_DIR) if f.endswith('npx_filt.npy')}

print(f'NPX window: -{PRE_MS} to +{POST_MS} ms  ({WIN_LEN} samples at {NPX_FS} Hz)')
print(f'Cells with NPX data: {sorted(NPX_CELLS)}')

In [ ]:
# ── Load transition table ──────────────────────────────────────────────────────
df_transitions = pd.read_pickle(TRANSITION_PICKLE)

# Keep only cells with NPX data and strong temporal component
df_npx = df_transitions[df_transitions['cell_id'].isin(NPX_CELLS)].copy()
print(f'Transitions with NPX data: {len(df_npx)}')
print(df_npx[['cell_id','spike_feature','temporal_rho','transition_time_ms',
              'cluster_before','cluster_after']].to_string(index=False))

In [ ]:
# ── Extract EAP waveforms for each transition ─────────────────────────────────
results = {}  # {(cell_id, feature): {'wf_pre', 'wf_post', 'times_pre', 'times_post', ...}}

for _, row in df_npx.iterrows():
    cid  = row['cell_id']
    feat = row['spike_feature']
    t_tr = row['transition_time_ms']

    # Load NPX recording
    npx_path = NPX_DIR + f'{cid}npx_filt.npy'
    if not os.path.exists(npx_path):
        print(f'  {cid}: NPX file not found, skipping')
        continue
    npx = np.load(npx_path, mmap_mode='r').astype(float)

    # Load cluster df for spike times
    cluster_pkl = CLUSTER_PICKLE_DIR + f'{cid}_cluster_df.pkl'
    df_c = pd.read_pickle(cluster_pkl)
    df_c = df_c.sort_values('spk_times_ms').reset_index(drop=True)

    pre_mask  = df_c['spk_times_ms'] <  t_tr
    post_mask = df_c['spk_times_ms'] >= t_tr

    t_pre  = df_c.loc[pre_mask,  'spk_times_ms'].values
    t_post = df_c.loc[post_mask, 'spk_times_ms'].values

    wf_pre  = extract_eap_waveforms(npx, t_pre,  NPX_FS, PRE_SAMP, POST_SAMP)
    wf_post = extract_eap_waveforms(npx, t_post, NPX_FS, PRE_SAMP, POST_SAMP)

    print(f'{cid} {feat}: {len(wf_pre)} pre-waveforms  |  {len(wf_post)} post-waveforms')

    results[(cid, feat)] = dict(
        wf_pre=wf_pre, wf_post=wf_post,
        t_pre=t_pre, t_post=t_post,
        t_transition=t_tr,
        temporal_rho=row['temporal_rho'],
        cluster_before=row['cluster_before'],
        cluster_after=row['cluster_after'],
    )

## PCA on EAP waveforms pre vs post transition

For each (cell, feature) pair: fit PCA on all waveforms combined (pre + post), then project each
group separately. If the EAP shape genuinely changes at the transition, pre and post will separate
in PC space. If they overlap, the NPX waveform is stable and the transition in patch-clamp features
is not detectable extracellularly.

In [ ]:
time_axis_ms = np.linspace(-PRE_MS, POST_MS, WIN_LEN)
n_cases = len(results)
n_cols  = min(n_cases, 3)
n_rows  = int(np.ceil(n_cases / n_cols)) * 3   # 3 rows per case: mean wf, PC1vs2, PC1 hist

fig = plt.figure(figsize=(n_cols * 5.5, n_cases * 9.5))

COLORS = {'pre': '#0072B2', 'post': '#D55E00'}

for case_idx, ((cid, feat), d) in enumerate(results.items()):
    wf_pre  = d['wf_pre']
    wf_post = d['wf_post']
    rho     = d['temporal_rho']
    cl_b    = d['cluster_before']
    cl_a    = d['cluster_after']
    t_tr_s  = d['t_transition'] / 1000.0

    if len(wf_pre) < 5 or len(wf_post) < 5:
        continue

    all_wf = np.vstack([wf_pre, wf_post])

    # Z-score each waveform independently (shape comparison)
    scaler = StandardScaler()
    all_wf_z = scaler.fit_transform(all_wf.T).T  # z-score per waveform

    pca = PCA(n_components=min(10, all_wf_z.shape[0] - 1))
    pcs = pca.fit_transform(all_wf_z)

    pc_pre  = pcs[:len(wf_pre)]
    pc_post = pcs[len(wf_pre):]

    # Mann-Whitney on PC1
    stat, p_mw = mannwhitneyu(pc_pre[:, 0], pc_post[:, 0], alternative='two-sided')
    p_str = '< 0.0001' if p_mw < 0.0001 else f'{p_mw:.4f}'
    stars = '***' if p_mw < 0.001 else ('**' if p_mw < 0.01 else ('*' if p_mw < 0.05 else 'ns'))

    row_base = case_idx * 3

    # ── Panel 1: mean EAP waveform pre vs post ──
    ax1 = fig.add_subplot(n_cases * 3, n_cols, row_base * n_cols + case_idx % n_cols + 1)
    for label, wf, col in [('pre',  wf_pre,  COLORS['pre']),
                             ('post', wf_post, COLORS['post'])]:
        m = np.mean(wf, axis=0)
        s = np.std(wf, axis=0) / np.sqrt(len(wf))
        ax1.plot(time_axis_ms, m, color=col, lw=2, label=f'{label} (n={len(wf)})')
        ax1.fill_between(time_axis_ms, m - s, m + s, color=col, alpha=0.25)
    ax1.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax1.set_xlabel('Time (ms)', fontsize=9)
    ax1.set_ylabel('Amplitude (a.u.)', fontsize=9)
    ax1.set_title(f'{cid}  {feat}\nρ={rho:+.2f}  {cl_b}→{cl_a}  |  transition at {t_tr_s:.0f}s',
                  fontsize=9, fontweight='bold')
    ax1.legend(fontsize=8, frameon=False)
    sns.despine(ax=ax1)

    # ── Panel 2: PC1 vs PC2 ──
    ax2 = fig.add_subplot(n_cases * 3, n_cols, (row_base + 1) * n_cols + case_idx % n_cols + 1)
    ax2.scatter(pc_pre[:, 0],  pc_pre[:, 1],  color=COLORS['pre'],  alpha=0.4, s=8, label='pre')
    ax2.scatter(pc_post[:, 0], pc_post[:, 1], color=COLORS['post'], alpha=0.4, s=8, label='post')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=9)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=9)
    ax2.set_title('EAP PCA', fontsize=9)
    ax2.legend(fontsize=8, frameon=False)
    sns.despine(ax=ax2)

    # ── Panel 3: PC1 histogram ──
    ax3 = fig.add_subplot(n_cases * 3, n_cols, (row_base + 2) * n_cols + case_idx % n_cols + 1)
    ax3.hist(pc_pre[:, 0],  bins=30, color=COLORS['pre'],  alpha=0.6, density=True, label='pre')
    ax3.hist(pc_post[:, 0], bins=30, color=COLORS['post'], alpha=0.6, density=True, label='post')
    ax3.set_xlabel('PC1', fontsize=9)
    ax3.set_ylabel('Density', fontsize=9)
    ax3.set_title(f'PC1 distribution  |  MW p = {p_str}  {stars}', fontsize=9)
    ax3.legend(fontsize=8, frameon=False)
    sns.despine(ax=ax3)

fig.suptitle('EAP waveform PCA: pre vs post temporal transition in patch-clamp cluster membership\n'
             'Blue = pre-transition spikes  |  Orange = post-transition spikes',
             fontsize=12, y=1.002)
fig.tight_layout()
plt.show()

## Summary

For each (cell, feature) pair:
- **Mean EAP waveform**: if pre and post waveforms differ visually, the NPX records the same state change as the patch-clamp
- **PC1 vs PC2**: separation indicates the EAP waveform changed at the transition point
- **PC1 histogram + Mann-Whitney**: statistical test of whether the PC1 distributions differ

Cells where the EAP PCA shows significant pre/post separation are candidates where the temporal
component reflects a *detectable extracellular* waveform change — supporting a genuine biological
state change rather than purely an artefact of the patch-clamp recording.